## Install dependencies

In [1]:
!pip install -q \
    "numpy>=1.22,<2.1" \
    "langchain==0.3.25" \
    "langchain-core==0.3.62" \
    "langchain-text-splitters==0.3.8" \
    "langchain-google-genai==2.1.5" \
    "langchain-chroma==0.2.4" \
    "langchain-huggingface==0.1.2" \
    "chromadb>=1.0.9" \
    "rank-bm25==0.2.2" \
    "sentence-transformers>=2.6.0" \
    "gradio>=4.40,<6.0" \
    "gitpython>=3.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.4/438.4 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 94.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 9.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━

## Configure Gemini API Key

In [2]:
## Setup api keys
import os

# Suppress ChromaDB telemetry warnings (harmless posthog SDK incompatibility)
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY_IMPL"] = "none"

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab secrets ✓")
except Exception:
    if "GOOGLE_API_KEY" not in os.environ:
        from getpass import getpass
        os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")
    print("API key set ✓")

API key set ✓


## Temporary store git repo

In [3]:
#Import codebases from git
import shutil
from pathlib import Path
import git

# Change this to point at your own repo any time.
REPO_URL = "https://github.com/Priyabrata111/Ecommerce-App"
LOCAL_PATH = Path("/tmp/repo")

# Wipe any previous clone
if LOCAL_PATH.exists():
    shutil.rmtree(LOCAL_PATH)

git.Repo.clone_from(REPO_URL, LOCAL_PATH, depth=1)
print(f"Cloned into {LOCAL_PATH}")
print(f"Files: {sum(1 for _ in LOCAL_PATH.rglob('*') if _.is_file())}")

Cloned into /tmp/repo
Files: 107


## View the Directory

In [4]:
def tree(p: Path, prefix: str = "", depth: int = 2) -> None:
    if depth < 0:
        return
    entries = sorted([e for e in p.iterdir() if not e.name.startswith(".")])
    for i, entry in enumerate(entries):
        is_last = i == len(entries) - 1
        connector = "└── " if is_last else "├── "
        print(prefix + connector + entry.name)
        if entry.is_dir():
            extension = "    " if is_last else "│   "
            tree(entry, prefix + extension, depth - 1)

tree(LOCAL_PATH)

├── client
│   └── build
│       ├── asset-manifest.json
│       ├── favicon.ico
│       ├── images
│       ├── index.html
│       ├── logo192.png
│       ├── logo512.png
│       ├── manifest.json
│       ├── robots.txt
│       └── static
├── config
│   └── db.js
├── controllers
│   ├── authController.js
│   ├── categoryController.js
│   └── productController.js
├── helpers
│   └── authHelper.js
├── middlewares
│   └── authMiddleware.js
├── models
│   ├── categoryModel.js
│   ├── orderModel.js
│   ├── productModel.js
│   └── userModel.js
├── package-lock.json
├── package.json
├── routes
│   ├── authRoute.js
│   ├── categoryRoutes.js
│   └── productRoutes.js
└── server.js


## Work with mutiple repos

In [5]:
REPOS = [
    {
        "name": "Ecommerce-App",
        "url": "https://github.com/Priyabrata111/Ecommerce-App"
    },
    {
        "name": "systemC",
        "url": "https://github.com/Priyabrata111/systemC"
    },
    {
        "name": "genAI_Learning",
        "url": "https://github.com/Priyabrata111/GenAI_Learning"
    },
    {
        "name": "Simon-Game",
        "url": "https://github.com/Priyabrata111/Simon-Game-Chellange"
    },
    {
        "name": "inotebook",
        "url": "https://github.com/Priyabrata111/inotebook"
    },
]


## Import all Repos in local system

In [6]:
#Import codebases from git
import shutil
from pathlib import Path
import git


## Store here
ROOT_DIR = Path("/tmp/repos")

for repo in REPOS:
    repo_path = ROOT_DIR / repo["name"]

    if repo_path.exists():
        shutil.rmtree(repo_path)

    git.Repo.clone_from(
        repo["url"],
        repo_path,
        depth=1
    )

## Check if repos are copied correctly

In [7]:
from pathlib import Path

for p in Path("/tmp/repos").iterdir():
    print(p)
    tree(p)

/tmp/repos/systemC
├── UST
│   └── Basics
│       ├── AT_tlm.cpp
│       ├── event.cpp
│       ├── handShake.cpp
│       ├── interrupts_method.cpp
│       ├── interrupts_thread.cpp
│       ├── multiTransaction.cpp
│       ├── producer_consumer.cpp
│       ├── raceAround.cpp
│       ├── tlm.cpp
│       └── tlm2.cpp
└── vConnect
    ├── EthernetController
    │   ├── EthernetFrame.h
    │   ├── receiver.h
    │   ├── target.h
    │   ├── testBench.cpp
    │   └── vECU.h
    ├── LearningAll.cpp
    ├── LearningBasics.cpp
    ├── TLM
    │   ├── LT.h
    │   ├── Packet.h
    │   ├── mainLT.cpp
    │   ├── myCPU.h
    │   ├── myCPUPacket.h
    │   ├── myMemory.h
    │   ├── myMemoryPacket.h
    │   ├── testBench02.cpp
    │   └── testbench.cpp
    ├── designCounterMethod.cpp
    └── designCounterThread.cpp
/tmp/repos/Simon-Game
├── game.js
├── index.html
├── sounds
│   ├── blue.mp3
│   ├── green.mp3
│   ├── red.mp3
│   ├── wrong.mp3
│   └── yellow.mp3
└── styles.css
/tmp/repos/inotebook
├──

## Load the repos as langchain documents

In [8]:


from pathlib import Path
from typing import List
from langchain_core.documents import Document
# Extensions relevant for MERN / Node.js / React
CODE_EXTENSIONS = {
      # C/C++
    ".c",
    ".cc",
    ".cpp",
    ".cxx",
    ".h",
    ".hh",
    ".hpp",
    ".hxx",

    # SystemC / TLM
    ".tpp",

    # Python
    ".py",
    ".ipynb",

    # JavaScript / TypeScript
    ".js",
    ".jsx",
    ".ts",
    ".tsx",

    # Config / Docs
    ".json",
    ".yaml",
    ".yml",
    ".xml",
    ".toml",
    ".ini",
    ".md",
    ".txt",

    # Web
    ".css",
    ".html",
}

# Folders to ignore
SKIP_DIRS = {
    ".git",
    "__pycache__",
    "node_modules",
    "dist",
    ".next",
    ".cache",
}
# Function to load the git repo as langchain model

def load_repo_documents(repo_path: Path,repo_name: str) -> List[Document]:
    docs = []

    for path in repo_path.rglob("*"):

        # Skip directories
        if not path.is_file():
            continue

        # Skip unwanted folders
        if any(part in SKIP_DIRS for part in path.parts):
            continue

        # Skip unsupported file types
        if path.suffix.lower() not in CODE_EXTENSIONS:
            continue

        try:
            text = path.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            continue

        rel = path.relative_to(repo_path)

        docs.append(
            Document(
                page_content=text,
                metadata={
                "repo": repo_name,
                "source": str(rel),
                "extension": path.suffix,
},
            )
        )

    return docs


# Function call to load the repo as langchain documents
all_docs = []
for repo in REPOS:
    repo_path = ROOT_DIR / repo["name"]

    docs = load_repo_documents(
        repo_path,
        repo_name=repo["name"]
    )

    all_docs.extend(docs)

    print(f"Loaded {len(docs)} files from {repo['name']}")


print(f"\nTotal files loaded: {len(all_docs)}")

for d in all_docs[:10]:
    print(
        f"[{d.metadata['repo']}] "
        f"{d.metadata['source']:40s} "
        f"({len(d.page_content)} chars)"
    )

Loaded 24 files from Ecommerce-App
Loaded 28 files from systemC
Loaded 12 files from genAI_Learning
Loaded 3 files from Simon-Game
Loaded 28 files from inotebook

Total files loaded: 95
[Ecommerce-App] package-lock.json                        (135391 chars)
[Ecommerce-App] package.json                             (854 chars)
[Ecommerce-App] server.js                                (1440 chars)
[Ecommerce-App] helpers/authHelper.js                    (384 chars)
[Ecommerce-App] models/userModel.js                      (629 chars)
[Ecommerce-App] models/categoryModel.js                  (273 chars)
[Ecommerce-App] models/orderModel.js                     (496 chars)
[Ecommerce-App] models/productModel.js                   (697 chars)
[Ecommerce-App] middlewares/authMiddleware.js            (829 chars)
[Ecommerce-App] routes/authRoute.js                      (1344 chars)


## Perform smart chunking of the loaded docs

In [9]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    Language,
    MarkdownHeaderTextSplitter,
)

def chunk_smart(docs):

    py_splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.PYTHON,
        chunk_size=1500,
        chunk_overlap=150,
    )

    js_splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.JS,
        chunk_size=1500,
        chunk_overlap=150,
    )

    cpp_splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.CPP,
        chunk_size=1500,
        chunk_overlap=150,
    )

    md_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "h1"),
            ("##", "h2"),
            ("###", "h3"),
        ],
        strip_headers=False,
    )

    fallback = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
    )

    out = []

    for doc in docs:

        ext = doc.metadata.get("extension", "").lower()

        # Python
        if ext == ".py":
            for c in py_splitter.split_documents([doc]):
                c.metadata["chunk_type"] = "python"
                out.append(c)

        # Markdown
        elif ext == ".md":
            for c in md_splitter.split_text(doc.page_content):
                c.metadata = {
                    **doc.metadata,
                    **c.metadata,
                    "chunk_type": "markdown",
                }
                out.append(c)

        # JavaScript
        elif ext in [".js", ".jsx"]:
            for c in js_splitter.split_documents([doc]):
                c.metadata["chunk_type"] = "javascript"
                out.append(c)

        # TypeScript
        elif ext in [".ts", ".tsx"]:
            for c in js_splitter.split_documents([doc]):
                c.metadata["chunk_type"] = "typescript"
                out.append(c)

        # C / C++ / SystemC / TLM
        elif ext in [
            ".c",
            ".cc",
            ".cpp",
            ".cxx",
            ".h",
            ".hh",
            ".hpp",
            ".hxx",
        ]:
            for c in cpp_splitter.split_documents([doc]):
                c.metadata["chunk_type"] = "cpp"
                out.append(c)

        # Jupyter Notebook
        elif ext == ".ipynb":
            for c in fallback.split_documents([doc]):
                c.metadata["chunk_type"] = "notebook"
                out.append(c)

        # Everything else
        else:
            for c in fallback.split_documents([doc]):
                c.metadata["chunk_type"] = "text"
                out.append(c)

    return out



smart_chunks = chunk_smart(all_docs)

print(f"Created {len(smart_chunks)} chunks")

from collections import Counter

chunk_dist = Counter(
    c.metadata.get("chunk_type")
    for c in smart_chunks
)

print(f"By chunk type: {dict(chunk_dist)}")

Created 4620 chunks
By chunk type: {'text': 964, 'javascript': 1057, 'cpp': 39, 'notebook': 2544, 'python': 3, 'markdown': 13}


## Store the chunks in chroma DB & store in Google Drive

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Local embedding model
# Downloads once, then runs locally
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Quick smoke test
v = embeddings.embed_query("smoke test")
print(f"✓ Local embeddings loaded (vector dim = {len(v)})")


# Create Chroma vector database
vectorstore_v1 = Chroma.from_documents(
    documents=smart_chunks,
    embedding=embeddings,
    collection_name="multi_repo_v1",
    persist_directory="/content/drive/MyDrive/chroma_db",  # storing it in gDrive so that we can only use this db later
)

print(f"✓ Indexed {len(smart_chunks)} chunks")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Local embeddings loaded (vector dim = 384)
✓ Indexed 4620 chunks


In [12]:
# Later fetch the db from drive from here 
# vectorstore_v1= Chroma(
#     collection_name="multi_repo_v1",
#     persist_directory="/content/drive/MyDrive/chroma_db",
#     embedding_function=embeddings
# )

##  Hybrid Search = semantic + BM25

In [13]:
# Build BM25 Index

from rank_bm25 import BM25Okapi
import re


def tokenize(text: str) -> list[str]:
    # Split on non-alphanumerics and lowercase. Keeps identifiers like 'JWTHandler' as one token.
    return [t.lower() for t in re.findall(r"\w+", text)]


# BM25 needs the tokenized chunks plus the original Document objects to return.
chunk_tokens = [tokenize(c.page_content) for c in smart_chunks]
bm25 = BM25Okapi(chunk_tokens)
print(f"BM25 ready over {len(smart_chunks)} chunks")

BM25 ready over 4620 chunks


## Reciprocal Rank Fusion

In [15]:


def hybrid_search(query: str, k: int = 4, fetch_k: int = 12) -> list:
    # Semantic results (already ordered by similarity).
    semantic = vectorstore_v1.similarity_search(query, k=fetch_k)

    # BM25 results.
    bm25_scores = bm25.get_scores(tokenize(query))
    top_bm25_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:fetch_k]
    keyword = [smart_chunks[i] for i in top_bm25_idx]

    # Reciprocal rank fusion — sum 1/(rank+60) across both lists.
    scores: dict[str, float] = {}
    docs_by_id: dict[str, object] = {}

    def doc_id(d):
        return f"{d.metadata.get('source','')}::{hash(d.page_content[:100])}"

    for rank, d in enumerate(semantic):
        did = doc_id(d)
        scores[did] = scores.get(did, 0) + 1 / (rank + 60)
        docs_by_id[did] = d
    for rank, d in enumerate(keyword):
        did = doc_id(d)
        scores[did] = scores.get(did, 0) + 1 / (rank + 60)
        docs_by_id[did] = d

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return [docs_by_id[did] for did, _ in ranked]


# Quick visual test
hits = hybrid_search("How Initiator makes a connection to target in TLM 2.0?", k=4)
for i, d in enumerate(hits, 1):
    print(f"{i}. {d.metadata['source']}")

1. UST/Basics/tlm2.cpp
2. UST/Basics/multiTransaction.cpp
3. UST/Basics/AT_tlm.cpp
4. vConnect/TLM/myCPUPacket.h


## Cross encoder reranker

In [16]:
## Load the reranker

from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded ✓")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded ✓


In [17]:
def search_with_rerank(query: str, k: int = 4, fetch_k: int = 12) -> list:
    candidates = hybrid_search(query, k=fetch_k, fetch_k=fetch_k * 2)
    if not candidates:
        return []

    pairs = [(query, d.page_content) for d in candidates]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:k]]



In [18]:
## Prompts & gemini
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

prompt = ChatPromptTemplate.from_template('''You are a helpful coding assistant answering questions about a codebase.
Use ONLY the context below. If the answer isn\'t in the context, say "I don\'t see that in the code."

Context:
{context}

Question: {question}

Answer:''')

def format_context(docs):
    return "\n\n---\n\n".join(
        f"# File: {d.metadata['source']}\n{d.page_content}" for d in docs
    )


def show_retrieval(question, docs):
    """Print which chunks were retrieved. Call before generation so students see retrieval as a step."""
    print(f"📚 Retrieved {len(docs)} chunks for: {question!r}")
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "?")
        size = len(d.page_content)
        line_info = ""
        if "start_line" in d.metadata:
            line_info = f":{d.metadata['start_line']}-{d.metadata['end_line']}"
        print(f"  {i}. {src}{line_info} ({size} chars)")
    print()

def ask_v1(question: str, k: int = 4) -> str:
    docs = search_with_rerank(question, k=k)
    show_retrieval(question, docs)
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"context": format_context(docs), "question": question})


print(ask_v1("How Initiator makes a connection to target in TLM 2.0?"))

📚 Retrieved 4 chunks for: 'How Initiator makes a connection to target in TLM 2.0?'
  1. UST/Basics/tlm.cpp (1495 chars)
  2. UST/Basics/multiTransaction.cpp (1336 chars)
  3. UST/Basics/AT_tlm.cpp (1491 chars)
  4. UST/Basics/tlm2.cpp (880 chars)

In the provided code, the Initiator makes a connection to the Target using the `bind` method of their respective TLM sockets.

For example, in `UST/Basics/tlm.cpp`, the connection is made in `sc_main` as follows:
```cpp
  Initiator init("init");
  Target tg("tg");
  
  init.sc1.bind(tg.sc2);
```
Here, `init.sc1` is the initiator socket and `tg.sc2` is the target socket.

Similarly, in `UST/Basics/multiTransaction.cpp`:
```cpp
  Initiator init("init");
  Target tg("tg");
  
  init.sc2.bind(tg.sc1);
```
And in `UST/Basics/tlm2.cpp`:
```cpp
  Initiator init("init");
  Target tg("tg");
  
  init.sc1.bind(tg.sc1);
```
